# Cross-Dataset Questions

Questions that need several datasets at once. Each of these is a hand-written
join, and each reports which datasets it could reach — so a missing section
reads as missing rather than as zero.

All of it depends on the team registry to reconcile how differently these
sources spell school names.

Covers: `team_profile`, `player_profile`, `draft_yield`,
`dollars_per_draft_pick`, `conference_report`, `pipeline`, `compare_teams`

In [1]:
from ncaa_bbStats import *

## One program, one season, every dataset

In [2]:
profile = team_profile("Tennessee", 2024)

identity = profile["identity"]
print(f"{identity['canonical_name']}  ({identity['conference']}, "
      f"Division {identity['division']})")
print(f"  IPEDS {identity['ipeds_unitid']} -- {identity['institution_name']}")

record = profile["record"]
print(f"\n  record    {record['wins']}-{record['losses']}  ({record['win_pct']})")
print(f"  RPI       {profile['rpi']['rpi_rank']}  "
      f"(SOS {profile['rpi']['sos_rank']})")
print(f"  Q1        {profile['rpi']['q1_wins']}-{profile['rpi']['q1_losses']}")
print(f"  budget    {profile['finance']['budget_pct']:.3f} percentile")
print(f"  roster    {profile['finance']['roster_size']:.0f}")
print(f"  drafted   {profile['draft']['picks']} players")
print(f"  pythag    expected {profile['pythagorean']['expected_win_pct']}, "
      f"actual {profile['pythagorean']['actual_win_pct']}")

Tennessee  (SEC, Division 1)
  IPEDS 221759 -- The University of Tennessee-Knoxville

  record    60-13  (0.822)
  RPI       1  (SOS 12)
  Q1        26-10
  budget    0.996 percentile
  roster    51
  drafted   8 players
  pythag    expected 0.807, actual 0.822


In [3]:
# Which datasets were reachable
print(profile["coverage"])

print("\nThe draft class:")
for pick in profile["draft"]["selections"]:
    bonus = f"${pick['signing_bonus']:,}" if pick["signing_bonus"] else "-"
    print(f"  #{pick['pick']:>3}  {pick['name']:24s} {pick['position']:4s} {bonus}")

{'stats': True, 'rpi': True, 'finance': True, 'draft': True}

The draft class:
  #  8  Christian Moore          2B   $4,997,500
  # 34  Blake Burke              1B   $2,100,000
  # 60  Billy Amick              3B   $1,453,700
  # 65  Dylan Dreiling           OF   $1,287,600
  # 76  Drew Beam                P    $1,097,500
  #134  Kavares Tears            OF   $525,200
  #138  A.J. Causey              P    $477,500
  #229  Aaron Combs              P    $247,500


In [4]:
# A Division III program has no RPI or draft data -- and says so
profile = team_profile("Amherst", 2015)
if profile:
    print(profile["identity"]["canonical_name"], profile["identity"]["division"])
    print("coverage:", profile["coverage"])
    print("rpi is None:", profile["rpi"] is None)

Amherst 3
coverage: {'stats': True, 'rpi': False, 'finance': False, 'draft': False}
rpi is None: True


## One player, every dataset

In [5]:
player = player_profile("Kade Anderson")
print(f"{player['name']}  ({player['role']})")
print(f"  seasons played : {player['seasons_played']}")
print(f"  program        : {player['team']['canonical_name']} "
      f"({player['team']['conference']})")

pitching = player["pitching"]
print(f"\n  {player['season']}: {pitching['so']:.0f} K in {pitching['ip']} IP, "
      f"ERA {pitching['era']:.2f}, cFIP {pitching['cfip']:.2f}")

draft = player["draft"]
print(f"\n  drafted #{draft['pick']} by the {draft['team_name']}")
print(f"  bonus ${draft['signing_bonus']:,} against a ${draft['slot_value']:,} slot")
print(f"  pre-draft rank: {player['prospect_rank']}")
print(f"\n  coverage: {player['coverage']}")

Kade Anderson  (pitcher)
  seasons played : [2024, 2025]
  program        : LSU (SEC)

  2025: 180 K in 119.0 IP, ERA 3.18, cFIP 4.00

  drafted #3 by the Seattle Mariners
  bonus $8,800,000 against a $9,504,400 slot
  pre-draft rank: 2

  coverage: {'batting': False, 'pitching': True, 'team': True, 'draft': True}


## Programs as talent pipelines

In [6]:
for team in ["LSU", "Vanderbilt", "Tennessee", "Northeastern"]:
    result = draft_yield(team, 2021, 2026)
    print(f"  {result['team']:16s} {result['picks']:3d} picks "
          f"({result['picks_per_year']:.1f}/yr)  "
          f"{result['first_round_picks']} first-rounders  "
          f"${result['total_bonus_dollars']:>12,}")

  LSU               45 picks (7.5/yr)  6 first-rounders  $  65,302,650
  Vanderbilt        38 picks (6.3/yr)  4 first-rounders  $  29,312,200
  Tennessee         48 picks (8.0/yr)  10 first-rounders  $  60,024,505
  Northeastern      16 picks (2.7/yr)  0 first-rounders  $   2,898,200


In [7]:
result = draft_yield("LSU", 2021, 2026)
print("LSU, year by year:")
for season, row in sorted(result["by_year"].items()):
    print(f"  {season}  {row['picks']:2d} picks  ${row['bonus_dollars']:>11,}")
print(f"\n  best pick: #{result['best_pick']['pick']} "
      f"{result['best_pick']['name']} ({result['best_pick']['year']})")

LSU, year by year:
  2021   2 picks  $  2,457,300
  2022   4 picks  $  7,344,300
  2023  13 picks  $ 23,905,950
  2024   9 picks  $  8,324,200
  2025   9 picks  $ 14,495,800
  2026   8 picks  $  8,775,100

  best pick: #1 Paul Skenes (2023)


### Budget in, prospects out

Budget is a percentile rather than dollars, so this is not a literal cost per
pick — it pairs spending rank with draft output, which is the comparison that
travels across seasons.

In [8]:
for team in ["Tennessee", "LSU", "Vanderbilt", "Coastal Carolina",
             "Northeastern", "Utah Tech"]:
    row = dollars_per_draft_pick(team, 2021, 2025)
    budget = f"{row['mean_budget_pct']:.3f}" if row["mean_budget_pct"] else "  n/a"
    print(f"  {row['team']:18s} budget {budget}  ->  "
          f"{row['picks_per_year']:.1f} picks/yr  "
          f"(${row['bonus_dollars_per_year']:>11,.0f}/yr)")

  Tennessee          budget 0.995  ->  8.4 picks/yr  ($ 11,151,241/yr)
  LSU                budget 0.998  ->  7.4 picks/yr  ($ 11,305,510/yr)
  Vanderbilt         budget 0.999  ->  6.6 picks/yr  ($  5,766,440/yr)
  Coastal Carolina   budget 0.977  ->  2.6 picks/yr  ($  1,602,160/yr)


  Northeastern       budget 0.947  ->  2.8 picks/yr  ($    567,140/yr)
  Utah Tech          budget 0.870  ->  0.0 picks/yr  ($          0/yr)


## Conferences

In [9]:
report = conference_report("SEC", 2025)
print(f"{report['conference']} {report['season']}")
print(f"  programs      : {report['programs']}")
print(f"  draft picks   : {report['draft_picks']}")
print(f"  bonus dollars : ${report['bonus_dollars']:,}")
print(f"  median budget : {report['median_budget_pct']}")
print(f"  best RPI      : {report['best_rpi_rank']}")
print(f"\n  standings:")
for row in report["standings"][:8]:
    print(f"    RPI {row['rpi_rank']:>3}  {row['team']:22s} "
          f"{row['wins']:>2}-{row['losses']:<2}")

SEC 2025
  programs      : 16
  draft picks   : 107
  bonus dollars : $95,003,505
  median budget : 0.9919
  best RPI      : 1

  standings:
    RPI   1  Arkansas               50-15
    RPI   3  Vanderbilt             43-18
    RPI   4  LSU                    53-15
    RPI   5  Auburn                 41-20
    RPI   6  Texas                  44-14
    RPI   7  Georgia                43-17
    RPI  12  Ole Miss               43-21
    RPI  14  Tennessee              46-19


## A program's trajectory

In [10]:
print("Coastal Carolina, 2021-2026:")
print(f"  {'yr':4s} {'conf':10s} {'W-L':>7s} {'RPI':>5s} {'budget':>8s} {'picks':>6s}")
for row in pipeline("Coastal Carolina"):
    print(f"  {row['season']} {row['conference']:10s} "
          f"{str(row['wins']) + '-' + str(row['losses']):>7s} "
          f"{str(row['rpi_rank']):>5s} "
          f"{row['budget_pct']:>8.3f} {row['draft_picks']:>6d}")

Coastal Carolina, 2021-2026:
  yr   conf           W-L   RPI   budget  picks
  2021 Sun Belt     27-24    96    0.979      2
  2022 Sun Belt     39-20    29    0.974      3
  2023 Sun Belt     42-21    19    0.975      1
  2024 Sun Belt     36-25    34    0.975      3
  2025 Sun Belt     56-13     2    0.983      4
  2026 Sun Belt     37-23    30    0.983      3


## Side by side

In [11]:
rows = compare_teams(["Tennessee", "LSU", "Coastal Carolina",
                      "Northeastern", "Utah Tech"], 2025)
print(f"  {'team':18s} {'conf':10s} {'W-L':>7s} {'RPI':>5s} {'budget':>8s} {'picks':>6s}")
for row in rows:
    print(f"  {row['team']:18s} {str(row['conference']):10s} "
          f"{str(row['wins']) + '-' + str(row['losses']):>7s} "
          f"{str(row['rpi_rank']):>5s} "
          f"{row['budget_pct']:>8.3f} {row['draft_picks']:>6d}")

  team               conf           W-L   RPI   budget  picks
  Tennessee          SEC          46-19    14    1.000      9
  LSU                SEC          53-15     4    0.999      9
  Coastal Carolina   Sun Belt     56-13     2    0.983      4
  Northeastern       CAA          49-11    26    0.944      5
  Utah Tech          WAC          24-31   250    0.869      0


## Recipe: draft picks per dollar spent

The kind of question that needs three datasets and the registry to connect them.

In [12]:
picks = {row["conference"]: row["picks"] for row in conference_draft_counts(2025)}

print(f"  {'conference':16s} {'budget':>8s} {'picks':>6s}")
for row in conference_spending(2025)[:12]:
    count = picks.get(row["conference"], 0)
    print(f"  {row['conference']:16s} {row['median_budget_pct']:>8.3f} {count:>6d}")

  conference         budget  picks


  SEC                 0.992    107
  DI Independent      0.986      7
  ACC                 0.981     61
  Big 12              0.976     57
  Big Ten             0.973     34
  The American        0.949     17
  WCC                 0.945      7
  Mountain West       0.938      7
  Sun Belt            0.933     21
  Big West            0.931     14
  Big East            0.925      6
  CUSA                0.917     14
